In [ ]:
#import packages
from obspy import read

import matplotlib.pyplot as plt
import glob
import os

import pandas as pd

import glob
import os

import matplotlib.pylab as plt
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
plt.style.use("ggplot")
%matplotlib inline

from msnoise.api import connect, get_results, build_movstack_datelist, get_params, get_t_axis, get_results_all, update_station

In [9]:
stations = ["XA.GP3", "XA.GP4"]
channels = ["Z"]

st = read("./DATA/MSEED/XA.GP3..Z.1976.263")
print(st)

st = read("./DATA/MSEED/XA.GP3..Z.1976.293")
print(st)

1 Trace(s) in Stream:
XA.G3..Z | 1976-09-19T00:00:00.001000Z - 1976-09-19T23:16:30.829615Z | 117.8 Hz, 9868473 samples
1 Trace(s) in Stream:
XA.G3..Z | 1976-10-19T00:00:00.001000Z - 1976-10-19T21:15:26.024867Z | 117.8 Hz, 9012860 samples


In [ ]:
! msnoise db init --tech=1

In [ ]:
! msnoise config set data_folder=./DATA/MSEED
! msnoise config set startdate=1976-09-19
! msnoise config set enddate=1976-10-19

In [ ]:
! msnoise -t 4 scan_archive --init --path ./DATA/MSEED
! msnoise populate --fromDA

In [ ]:
! msnoise info

In [ ]:
# open in terminaL: msnoise admin

In [ ]:
# plot data availability
from msnoise.plots.data_availability import main
main(chan="Z", show=True)

In [ ]:
# data size for different station per day
for station in stations:
    for channel in channels:
        files = sorted(glob.glob("DATA/MSEED/{}.*.{}*".format(station, channel)))
        if not len(files):
            continue
        days = []
        sizes = []
        for file in files:
            st = read(file, headlonly=True)
            
            sizes.append(os.path.getsize(file) / 1024)
            days.append((st[0].stats.starttime + 60*60*24).date)
        plt.scatter(days, sizes, label="{}.{}".format(station, channel))
plt.legend(loc=4, ncols=2)
plt.title("Size of measurment data")
plt.ylabel("kB")
plt.show()

In [ ]:
! msnoise new_jobs --init
! msnoise info -j

In [ ]:
#! msnoise db execute "insert into filters (ref, low, mwcs_low, high, mwcs_high, mwcs_wlen, mwcs_step, used) values (2, 0.1, 0.1, 1.0, 1.0, 12.0, 4.0, 1)"
#! msnoise db execute "insert into filters (ref, low, mwcs_low, high, mwcs_high, mwcs_wlen, mwcs_step, used) values (3, 1, 1, 2, 2, 12.0, 4.0, 1)"
#! msnoise db execute "insert into filters (ref, low, mwcs_low, high, mwcs_high, mwcs_wlen, mwcs_step, used) values (4, 2, 2, 4, 4, 12.0, 4.0, 1)"
#! msnoise db execute "insert into filters (ref, low, mwcs_low, high, mwcs_high, mwcs_wlen, mwcs_step, used) values (5, 4, 4, 8, 8, 2.0, 1.0, 1)"
! msnoise db execute "insert into filters (ref, low, mwcs_low, high, mwcs_high, mwcs_wlen, mwcs_step, used) values (1, 3, 3, 30, 30, 2.0, 1.0, 1)"

In [ ]:
# Reset jobs
! msnoise reset CC --all
! msnoise reset STACK --all
! msnoise reset MWCS --all
! msnoise reset DTT --all

In [ ]:
! msnoise config set resampling_method=Lancoz
! msnoise config set cc_sampling_rate=115
! msnoise config set preprocess_highpass=1
! msnoise config set preprocess_lowpass=40
# ! msnoise -t 6 -d 1 -v cc compute_cc  # run in console

In [ ]:
! msnoise config sync
#! msnoise -t 6 cc stack -r  # run in console #if error occurs, delete STACK2 directory

In [ ]:
! msnoise reset STACK 
! msnoise config set mov_stack='(("1d","1d"),("2d","1d"),("5d","1d"))'
#! msnoise -t 6 cc stack -m  # run in console

In [ ]:
# connect to the database
db = connect()

# Obtain a list of dates between ``start_date`` and ``enddate``
start, end, datelist = build_movstack_datelist(db)

# Get the list of parameters from the DB:
params = get_params(db)

# Get the time axis for plotting the CCF:
taxis = get_t_axis(db)

filter_id = 1
mov_stack = 1

st_1 = read("STACKS/01/001_DAYS/ZZ/XA.G1.--_XA.G2.--/1976-09-10.MSEED")
print(st_1)

# Fileformat specific information is stored here.
print(st_1[0].stats.mseed)
st_1.plot()



In [ ]:
# connect to the database
db = connect()

# Obtain a list of dates between ``start_date`` and ``enddate``
start, end, datelist = build_movstack_datelist(db)

# Get the list of parameters from the DB:
params = get_params(db)

# Get the time axis for plotting the CCF:
taxis = get_t_axis(db)

filter_id = 1
mov_stack = 1

# Get the results for two station, filter id=1, ZZ component, mov_stack=1 and the results as a 2D array:
n_stack, ccf_stack = get_results(db, "XA.G2.--", "XA.G3.--", filter_id, "ZZ", datelist,
                      mov_stack, format="stack", params=params)

print(taxis)

plt.figure()
plt.plot(taxis, ccf_stack)
plt.title("Reference Function")
plt.xlabel("Lag Time (s)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
# Get the results for two station, filter id=1, ZZ component, mov_stack=1 and the results as a 2D array:
n, ccfs = get_results(db, "XA.G1.--", "XA.G2.--", filter_id, "ZZ", datelist,
                      mov_stack, format="matrix", params=params)

# Convert to a pandas DataFrame object for convenience, and drop empty rows:
df = pd.DataFrame(ccfs, index=pd.DatetimeIndex(datelist), columns=taxis)
df = df.dropna()


# Define the 99% percentile of the data, for visualisation purposes:
clim = df.mean(axis="index").quantile(0.999)

fig, ax = plt.subplots(figsize=(12,8))
plt.pcolormesh(df.columns, df.index.to_pydatetime(), df.values,
               vmin=-clim, vmax=clim, rasterized=True, cmap="RdBu")
plt.colorbar()
plt.title("Interferogram")
plt.xlabel("Lag Time (s)")
plt.ylim(df.index[0],df.index[-1])
plt.xlim(df.columns[0], df.columns[-1])
plt.subplots_adjust(left=0.15)